In [1]:
!pip install -q ultralytics

import os, shutil, yaml, cv2, torch, pandas as pd
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

os.environ["WANDB_DISABLED"] = "true"

# ---------- Locate kit ----------
INPUT = Path("/kaggle/input")
candidates = list(INPUT.rglob("train_labels.csv"))
assert candidates, (
    "train_labels.csv nahi mila /kaggle/input ke andar. "
    "Right panel -> Add Data -> competition dataset attach karo."
)
KIT = candidates[0].parent
print("KIT =", KIT)
print("Kit files (first 25):")
for p in sorted(KIT.iterdir())[:25]:
    print("  ", p.name)

# ---------- Locate image folders (robust) ----------
def find_dir(names):
    for n in names:
        p = KIT / n
        if p.is_dir():
            return p
    return None

TRAIN_IMG_DIR = find_dir(["rain_images", "train_images", "train"])
VAL_IMG_DIR   = find_dir(["val_images", "validation_images", "val"])
TEST_IMG_DIR  = find_dir(["test_images", "test"])
assert TRAIN_IMG_DIR, "Training image folder nahi mila"
assert VAL_IMG_DIR,   "Val image folder nahi mila"
assert TEST_IMG_DIR,  "Test image folder nahi mila"

TRAIN_CSV    = KIT / "train_labels.csv"
VAL_CSV      = KIT / "val_labels.csv"
VAL_IDS_CSV  = KIT / "val_image_ids.csv"
TEST_IDS_CSV = KIT / "test_image_ids.csv"
for f in [TRAIN_CSV, VAL_CSV, VAL_IDS_CSV, TEST_IDS_CSV]:
    assert f.exists(), f"Missing: {f}"

print("TRAIN_IMG_DIR =", TRAIN_IMG_DIR)
print("VAL_IMG_DIR   =", VAL_IMG_DIR)
print("TEST_IMG_DIR  =", TEST_IMG_DIR)

# ---------- Work dir (fresh) ----------
WORK = Path("/kaggle/working/rdd_yolo")
if WORK.exists():
    shutil.rmtree(WORK)
for sub in ["images/train", "labels/train", "images/val", "labels/val"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def resolve_image(images_dir, image_id):
    image_id = str(image_id)
    p = images_dir / image_id
    if p.exists():
        return p
    for ext in IMG_EXTS:
        p = images_dir / (image_id + ext)
        if p.exists():
            return p
    stem = Path(image_id).stem
    for ext in IMG_EXTS:
        p = images_dir / (stem + ext)
        if p.exists():
            return p
    return None

def make_yolo_split(images_dir, label_csv, ids_csv, split):
    df = pd.read_csv(label_csv)
    boxes = defaultdict(list)
    for _, r in df.iterrows():
        boxes[str(r["image_id"])].append((
            int(r["class_id"]),
            float(r["x_min"]), float(r["y_min"]),
            float(r["x_max"]), float(r["y_max"]),
        ))
    ids_df = pd.read_csv(ids_csv)
    id_col = "image_id" if "image_id" in ids_df.columns else ids_df.columns[0]
    image_ids = ids_df[id_col].astype(str).tolist()

    n_ok = n_miss = 0
    for image_id in image_ids:
        img_path = resolve_image(images_dir, image_id)
        if img_path is None:
            n_miss += 1
            continue
        dst = WORK / "images" / split / img_path.name
        if not dst.exists():
            try:
                os.symlink(img_path.resolve(), dst)
            except FileExistsError:
                pass
        h, w = cv2.imread(str(img_path)).shape[:2]
        box_list = []
        for key in {image_id, img_path.name, img_path.stem}:
            box_list += boxes.get(key, [])
        lines = []
        for cls, x1, y1, x2, y2 in box_list:
            x1 = max(0.0, min(x1, w - 1)); x2 = max(0.0, min(x2, w - 1))
            y1 = max(0.0, min(y1, h - 1)); y2 = max(0.0, min(y2, h - 1))
            if x2 <= x1 or y2 <= y1:
                continue
            xc = ((x1 + x2) / 2) / w
            yc = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        (WORK / "labels" / split / (img_path.stem + ".txt")).write_text("\n".join(lines))
        n_ok += 1
    print(f"{split}: {n_ok} images written, {n_miss} missing")

make_yolo_split(TRAIN_IMG_DIR, TRAIN_CSV, TRAIN_CSV, "train")
make_yolo_split(VAL_IMG_DIR, VAL_CSV, VAL_IDS_CSV, "val")

yaml_path = WORK / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(
        {"path": str(WORK), "train": "images/train", "val": "images/val", "names": CLASS_NAMES},
        f, sort_keys=False
    )

print("Dataset ready:", yaml_path)
print("Train imgs:", len(list((WORK/'images/train').iterdir())))
print("Val imgs:  ", len(list((WORK/'images/val').iterdir())))





Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
KIT = /kaggle/input/competitions/datasprint-2-road-damage-detection
Kit files (first 25):
   README.md
   benchmark_speed.py
   check_submission.py
   inference.py
   make_kaggle_submission.py
   requirements.txt
   score.py
   test_image_ids.csv
   test_images
   train_baseline.py
   train_images
   train_labels.csv
   val_image_ids.csv
   val_images
   val_labels.csv
TRAIN_IMG_DIR = /kaggle/input/competitions/datasprint-2-road-damage-detection/train_images
VAL_IMG_DIR   = /kaggle/input/competitions/datasprint-2-road-damage-detection/val_images
TEST_IMG_DIR  = /kaggle/input/competitions/datasprint-2-road-damage-detection/test_images
train: 17517 images written, 0 missing
val: 2150 images written, 0

In [2]:
#cell 2
model = YOLO("yolov8n.pt")   # COCO-pretrained, allowed
model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/runs",
    name="baseline",
    exist_ok=True,
    patience=15,
    workers=2,
    verbose=True,
)

BEST_WEIGHTS = "/kaggle/working/runs/baseline/weights/best.pt"
print("Best weights:", BEST_WEIGHTS)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rdd_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=None, opset=None, optimize=False

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50       2.5G      2.061      2.615      1.889         33        640: 100% ━━━━━━━━━━━━ 533/533 7.3it/s 1:13
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 5.7it/s 12.0s
                   all       2150       2561      0.211      0.223      0.145     0.0525

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50       2.5G      2.067      2.591      1.891         51        640: 100% ━━━━━━━━━━━━ 533/533 7.2it/s 1:14
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 5.6it/s 12.2s
                   all       2150       2561      0.235      0.268      0.178     0.0642

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50       2.5G      2.028      2.499       1.85         43        640: 100% ━━━━━━━━━━━━ 533/533 7.1it/s 1:15
                 Class     Images  Instances      Box(

In [3]:
model = YOLO(BEST_WEIGHTS)

test_ids_df = pd.read_csv(TEST_IDS_CSV)
id_col = "image_id" if "image_id" in test_ids_df.columns else test_ids_df.columns[0]
all_test_ids = test_ids_df[id_col].astype(str).tolist()

name_to_id = {}
for tid in all_test_ids:
    p = resolve_image(TEST_IMG_DIR, tid)
    if p is not None:
        name_to_id[p.name] = tid
    name_to_id[tid] = tid

results = model.predict(
    source=str(TEST_IMG_DIR),
    stream=True,
    conf=0.001,          # required
    iou=0.5,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    max_det=300,
    verbose=False,
)
preds_by_id = defaultdict(list)
for res in results:
    image_id = name_to_id.get(Path(res.path).name, Path(res.path).name)
    if res.boxes is None or len(res.boxes) == 0:
        continue
    xyxy = res.boxes.xyxy.cpu().numpy()
    cls  = res.boxes.cls.cpu().numpy().astype(int)
    conf = res.boxes.conf.cpu().numpy()
    for (x1, y1, x2, y2), c, s in zip(xyxy, cls, conf):
        preds_by_id[image_id].append((int(c), float(s), float(x1), float(y1), float(x2), float(y2)))

sub_rows = []
for image_id in all_test_ids:
    rs = preds_by_id.get(image_id, [])
    if not rs:
        sub_rows.append({"image_id": image_id, "PredictionString": "none"})
        continue
    parts = [f"{c} {s:.6f} {x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f}"
             for c, s, x1, y1, x2, y2 in rs]
    sub_rows.append({"image_id": image_id, "PredictionString": " ".join(parts)})

submission = pd.DataFrame(sub_rows, columns=["image_id", "PredictionString"])
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("Rows:", len(submission))
print("File: /kaggle/working/submission.csv")



     image_id                                   PredictionString
0  img_000001  0 0.029865 379.55 468.77 399.96 503.79 2 0.025...
1  img_000009  0 0.422198 441.58 329.04 595.25 695.74 0 0.197...
2  img_000010  0 0.252184 497.28 472.43 592.85 547.56 1 0.016...
3  img_000012  2 0.518682 166.96 269.65 472.40 389.20 1 0.224...
4  img_000015  0 0.126149 402.70 450.12 551.67 567.71 1 0.033...
Rows: 4200
File: /kaggle/working/submission.csv
